# Chapter 7: Independent checks

*Code cells: MIT licence (`LICENSE`). Text and figures: CC BY 4.0 (`DATA_LICENSE.md`). Part of the code-and-data repository for "Mean span dimension of an ideal wormlike chain" (Y. Wang, 2026).*

In [1]:
import sys, json, math, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'tutorial' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'code'))
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt
pd.set_option('display.precision', 8)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
REF = pd.read_csv(ROOT / 'data' / 'wlc_mean_span_reference.csv')
LAM, F = REF['lambda'].to_numpy(), REF['F_continuum'].to_numpy()
Z_M, B_COEF, A_GAUSS = 0.71821164, 0.64680, 4/math.sqrt(3*math.pi)
def check(name, value, expected, tol):
    ok = abs(value - expected) <= tol
    print(('PASS' if ok else 'FAIL') + f'  {name}: {value:.10g}  (expected {expected:.10g}, tol {tol:g})')
    return ok

## Why two more calculations

The spectral solver discretizes the kinetic equation in one particular way. A second discretization of
the same equation (finite volumes) checks the solver; sampling chain conformations checks the kinetic
equation itself, since the chains never see the kinetic equation.

## Finite-volume solution

`code/wlc_finite_volume.py` uses a cell-centered grid in $(y,u)$, conservative angular fluxes, upwind
transport, the incoming ghost value 1 at $y=0$, and the exact compact support $q=0$ for $y>\lambda$; the
semidiscrete system is propagated in $\lambda$ by a sparse matrix exponential. We run a small grid live
to show the mechanism, then load the archived extrapolations.

In [2]:
from wlc_finite_volume import FVSettings, solve_fv
ref1 = float(REF.loc[REF['lambda']==1.0, 'F_continuum'].iloc[0])
for Ny, Nu in ((100, 16), (200, 24), (400, 32)):
    t0 = time.time(); v = solve_fv(1.0, FVSettings(Ny=Ny, Nu=Nu, Ymax=1.0))
    print(f'Ny={Ny:4d} Nu={Nu:3d}  F={v:.6f}  rel. diff from spectral {100*(v/ref1-1):+.3f} %   ({time.time()-t0:.1f} s)')
fv = pd.read_csv(ROOT/'data'/'finite_volume_comparison.csv')
print('\narchived extrapolations:'); print(fv[['lambda','F_spectral','F_continuum_quad','relative_difference_pct','FV_combined_sensitivity_rel_pct']].to_string(index=False))

Ny= 100 Nu= 16  F=0.458080  rel. diff from spectral +0.257 %   (0.0 s)
Ny= 200 Nu= 24  F=0.457646  rel. diff from spectral +0.162 %   (0.1 s)


Ny= 400 Nu= 32  F=0.457314  rel. diff from spectral +0.090 %   (0.3 s)

archived extrapolations:
 lambda  F_spectral  F_continuum_quad  relative_difference_pct  FV_combined_sensitivity_rel_pct
    0.1  0.04952372        0.04952939               0.01143637                       0.00583300
    1.0  0.45690535        0.45692649               0.00462824                       0.00546157
   10.0  2.88641074        2.88691926               0.01761780                       0.01210756


The coarse grids converge toward the spectral value from below; the archived extrapolations agree
within 0.018 % when the grids are extrapolated with the integer convergence orders of the scheme; at
$\lambda=0.1$ and 10 that exceeds the finite-volume sensitivity by factors of 2.0 and 1.5, and all three
residuals are positive. The extended grid (`code/build_finite_volume_extended_audit.py`, $N_y=1600$, $N_u=64$)
shows why: the observed spatial order (`code/finite_volume_order_analysis.py`) is 1.3-2.0 at $\lambda=0.1$ but
only 0.77-0.94 at $\lambda=1$ and 10, so the integer-power model leaves a positive bias. A free-order
(Aitken) extrapolation removes it: the residuals become +0.004, -0.003 and -0.003 %. Methods independent of
the spectral solver therefore corroborate the curve to about $4\times10^{-5}$ (finite volume) and
$4\times10^{-4}$-$9\times10^{-4}$ (path sampling, one standard error, including the long-chain runs at
$L/p=10^3$ and $10^4$ in `data/heat_path_long.csv` and `data/heat_path_long_pooled.csv`). At $L/p=100$ a separable
finite-volume extrapolation (`code/build_finite_volume_long.py`) agrees with the spectral value to $3\times10^{-5}$.

In [3]:
ext = pd.read_csv(ROOT/'data'/'finite_volume_extended_comparison.csv')
print(ext.to_string(index=False))
fvo = pd.read_csv(ROOT/'data'/'finite_volume_order_analysis.csv')
print(); print(fvo.to_string(index=False))

 lambda  F_spectral  F_FV_extended  relative_difference_pct  spatial_sensitivity_abs  angular_sensitivity_abs  combined_sensitivity_abs  combined_sensitivity_rel_pct
    0.1  0.04952372     0.04952801               0.00866183               0.00000366               0.00000005                0.00000371                    0.00748576
    1.0  0.45690535     0.45692085               0.00339247               0.00001629               0.00000043                0.00001671                    0.00365753
   10.0  2.88641074     2.88678958               0.01312499               0.00025149               0.00000030                0.00025179                    0.00872221

 lambda  observed_order_Ny_min  observed_order_Ny_max observed_orders_at_Nu64  F_FV_free_order  F_spectral  relative_difference_pct  angular_fit_change_pct  F_FV_order_trend  relative_difference_order_trend_pct  spatial_model_change_pct
    0.1             1.26288407             2.03334329             1.263;1.442       0.04952552  0.

## Heat-kernel path sampling

The archived campaign (`code/sphere_heat_path_validation.py`) used 40 000 chains at each of
$\lambda=0.1,1,10,100$ and four contour resolutions sharing the same tangent paths, so step-size effects
are paired differences. Chapter 1 ran a small version of the same sampler.

In [4]:
paths = pd.read_csv(ROOT/'data'/'heat_path_coupled.csv')
refmap = REF.set_index('lambda').F_continuum
fine = paths.sort_values('steps').groupby('lam').tail(1).copy()
fine['F_spec'] = fine.lam.map(refmap); fine['dev (%)'] = 100*(fine.span/fine.F_spec-1); fine['2SE (%)'] = 200*fine.span_se/fine.F_spec
print(fine[['lam','steps','n','span','span_se','F_spec','dev (%)','2SE (%)']].to_string(index=False))
print('\nall finest-grid estimates within 2 SE of the spectral values:', bool(np.all(np.abs(fine['dev (%)']) <= fine['2SE (%)'])))

  lam  steps     n        span    span_se      F_spec     dev (%)    2SE (%)
  0.1    512 40000  0.04952342 0.00002323  0.04952372 -0.00061542 0.09383221
  1.0   1024 40000  0.45675819 0.00019103  0.45690535 -0.03220789 0.08362026
 10.0   2048 40000  2.88613352 0.00241220  2.88641074 -0.00960434 0.16714166
100.0   8192 40000 11.65022939 0.01061953 11.65759274 -0.06316354 0.18219077

all finest-grid estimates within 2 SE of the spectral values: True


In [5]:
paths_long = pd.concat([pd.read_csv(ROOT/'data'/'heat_path_long.csv').query('lam < 5e3'),
                        pd.read_csv(ROOT/'data'/'heat_path_long_pooled.csv')])      # 1e4: four pooled batches
fl = paths_long.sort_values('steps').groupby('lam').tail(1).copy()
fl['F_spec'] = fl.lam.map(refmap); fl['dev/SE'] = (fl.span-fl.F_spec)/fl.span_se
print(fl[['lam','steps','n','span','span_se','F_spec','dev/SE']].to_string(index=False))

    lam  steps     n         span    span_se       F_spec     dev/SE
 1000.0  16384 40000  39.79830064 0.03539794  39.78660988 0.33026669
10000.0 131072 80000 128.88990885 0.07926909 128.86404808 0.32624026


Fig. S1 of the supplementary material (`pmf.figure_checks()`) plots these deviations, together with the
finite-volume results and the paired step-size differences. Sampling errors exclude heat-kernel truncation,
the inverse-CDF grid and between-node interpolation.

In [6]:
import plot_main_figures as pmf
fig = pmf.figure_checks(); fig

<Figure size 744.882x277.165 with 2 Axes>